# RAG Demo 2 — PDF Extractors & Section-Aware Chunking

A companion to `01_demo.ipynb`, zoomed in on **Stage 1 (document processing)** and **Stage 2 (chunker)**.

`01_demo` used `pymupdf` for plain-text extraction and fixed-size character windows for chunking. This notebook swaps both:

- **Stage 1 — compare three PDF extractors side by side:**
  - `extract_pdf_basic` — raw pymupdf text (the `01_demo` baseline; flat, no structure).
  - `extract_pdf_docling` — docling, layout-aware Markdown.
  - `extract_pdf_pymupdf4llm` — PyMuPDF4LLM Markdown.
  - Metadata is read separately via `extract_pdf_meta` (junk-filtered title/authors).
- **Stage 2 — section-aware chunking:** split the Markdown on headings with LangChain's `MarkdownHeaderTextSplitter`, instead of blind character windows.

Same folder-contract pipeline as `01_demo`; only the extractor and splitter change.

### Extra dependencies

Beyond `requirements.txt`, this notebook needs:

```bash
pip install docling pymupdf4llm langchain langchain-text-splitters
```

In [12]:
!pip install docling pymupdf4llm langchain langchain-text-splitters


  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 548.1/548.1 kB 3.7 MB/s eta 0:00:00
Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl (158 kB)
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 5.1
    Uninstalling PyYAML-5.1:
      Successfully uninstalled PyYAML-5.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
frontmatter 3.0.8 requires PyYAML==5.1, but you have pyyaml 6.0.3 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from document_processing.pdf_proc import (
    extract_pdf_basic,
    extract_pdf_docling,
    extract_pdf_pymupdf4llm,
    extract_pdf_meta,
)
from chunker import Chunk

In [5]:
RAW_DIR = ROOT_DIR / "data" / "raw"
MARKDOWN_DIR = ROOT_DIR / "data" / "markdown"
CHUNKS_DIR = ROOT_DIR / "data" / "chunks"
RAW_DIR.mkdir(parents=True, exist_ok=True)
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

print("raw dir:     ", RAW_DIR)
print("markdown dir:", MARKDOWN_DIR)
print("chunks dir:  ", CHUNKS_DIR)

raw dir:      c:\Users\User\Documents\bgu-ai\code\ai-rag-agents-course\rag\rag-demo\data\raw
markdown dir: c:\Users\User\Documents\bgu-ai\code\ai-rag-agents-course\rag\rag-demo\data\markdown
chunks dir:   c:\Users\User\Documents\bgu-ai\code\ai-rag-agents-course\rag\rag-demo\data\chunks


### Pick a sample PDF from `data/raw/`

Section-aware chunking only makes sense for PDFs (the markdown extractors emit headings). Drop a PDF into `data/raw/` if it's empty.

In [6]:
pdf_files = [p for p in sorted(RAW_DIR.iterdir()) if p.is_file() and p.suffix.lower() == ".pdf"]
assert pdf_files, f"No PDFs in {RAW_DIR}. Drop a .pdf and re-run."

sample_pdf = pdf_files[0]
print("sample PDF:", sample_pdf.name)

sample PDF: Large Language Models the New Scrum Masters.pdf


## Stage 1 — Three PDF extractors, side by side

### A. `extract_pdf_basic` — raw pymupdf text

Fast, no layout model. Returns a `(text, meta)` tuple where `text` is a flat blob with no Markdown structure — so there are no headings to chunk on later. This is what `01_demo` used.

In [7]:
basic_text, basic_meta = extract_pdf_basic(sample_pdf)
print(f"chars: {len(basic_text)}   meta: {basic_meta}")
print("--- first 800 chars ---")
print(basic_text[:800])

chars: 23223   meta: {'title': 'Large Language Models, the New Scrum Masters', 'authors': ['Juan Couder and Omar Ochoa']}
--- first 800 chars ---
Papers 
RED Innovation: Using Scrum to Develop an 
Agile Department 
2024 
Large Language Models, the New Scrum Masters 
Large Language Models, the New Scrum Masters 
Juan Couder 
ortizcoj@my.erau.edu 
Omar Ochoa 
ochoao@erau.edu 
Follow this and additional works at: https://commons.erau.edu/red-papers 
Scholarly Commons Citation 
Scholarly Commons Citation 
Couder, J., & Ochoa, O. (2024). Large Language Models, the New Scrum Masters. , (). Retrieved from 
https://commons.erau.edu/red-papers/2 
This Paper is brought to you for free and open access by the RED Innovation: Using Scrum to Develop an Agile 
Department at Scholarly Commons. It has been accepted for inclusion in Papers by an authorized administrator of 
Scholarly Commons. For more information, please contact commons@erau.edu. 





### B. `extract_pdf_docling` — docling, layout-aware Markdown

Runs docling's document model: reconstructs reading order, headings, and tables, then emits Markdown. Slowest of the three (first run downloads models), but the Markdown structure is exactly what makes section chunking possible. Returns a Markdown `str`.

In [8]:
docling_md = extract_pdf_docling(sample_pdf)
print(f"chars: {len(docling_md)}   heading-ish lines: {docling_md.count(chr(10) + '#')}")
print("--- first 800 chars ---")
print(docling_md[:800])

[INFO] 2026-05-19 17:00:31,918 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-19 17:00:32,013 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-19 17:00:32,015 [RapidOCR] main.py:57: Using C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-19 17:00:32,212 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-19 17:00:32,252 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-19 17:00:32,253 [Rapid

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

chars: 27983   heading-ish lines: 26
--- first 800 chars ---
<!-- image -->

[Papers](https://commons.erau.edu/red-papers)

2024

## Large Language Models, the New Scrum Masters

Juan Couder ortizcoj@my.erau.edu

Omar Ochoa ochoao@erau.edu

Follow this and additional works at: https://commons.erau.edu/red-papers

## Scholarly Commons Citation

Couder, J., &amp; Ochoa, O. (2024). Large Language Models, the New Scrum Masters. , (). Retrieved from https://commons.erau.edu/red-papers/2

This Paper is brought to you for free and open access by the RED Innovation: Using Scrum to Develop an Agile Department at Scholarly Commons. It has been accepted for inclusion in Papers by an authorized administrator of Scholarly Commons. For more information, please contact commons@erau.edu.

RED Innovation: Using Scrum to Develop an Agile Department See discussions, s


### C. `extract_pdf_pymupdf4llm` — PyMuPDF4LLM Markdown

Built on pymupdf, much faster than docling, infers `#` headings from font sizes. A good speed/structure middle ground. Returns a Markdown `str`.

In [9]:
mu4llm_md = extract_pdf_pymupdf4llm(sample_pdf)
print(f"chars: {len(mu4llm_md)}   heading-ish lines: {mu4llm_md.count(chr(10) + '#')}")
print("--- first 800 chars ---")
print(mu4llm_md[:800])

chars: 23830   heading-ish lines: 11
--- first 800 chars ---
**==> picture [216 x 52] intentionally omitted <==**

Papers 

RED Innovation: Using Scrum to Develop an Agile Department 

2024 

## **Large Language Models, the New Scrum Masters** 

Juan Couder ortizcoj@my.erau.edu 

Omar Ochoa ochoao@erau.edu 

Follow this and additional works at: https://commons.erau.edu/red-papers 

## **Scholarly Commons Citation** 

Couder, J., & Ochoa, O. (2024). Large Language Models, the New Scrum Masters. , (). Retrieved from https://commons.erau.edu/red-papers/2 

This Paper is brought to you for free and open access by the RED Innovation: Using Scrum to Develop an Agile Department at Scholarly Commons. It has been accepted for inclusion in Papers by an authorized administrator of Scholarly Commons. For more information, please contact commons@erau.edu. 

See 


### Side-by-side summary

`chars` ~ how much text was recovered; `headings` ~ how much section structure is available for Stage 2. `extract_pdf_basic` should show ~0 headings — that's the point: it can't be section-split.

In [10]:
def headings(md):
    return sum(1 for line in md.splitlines() if line.lstrip().startswith("#"))

print(f"{'extractor':<26}{'chars':>10}{'headings':>12}")
print(f"{'extract_pdf_basic':<26}{len(basic_text):>10}{headings(basic_text):>12}")
print(f"{'extract_pdf_docling':<26}{len(docling_md):>10}{headings(docling_md):>12}")
print(f"{'extract_pdf_pymupdf4llm':<26}{len(mu4llm_md):>10}{headings(mu4llm_md):>12}")

extractor                      chars    headings
extract_pdf_basic              23223           0
extract_pdf_docling            27983          26
extract_pdf_pymupdf4llm        23830          11


In [9]:
# write each extractor's output to data/markdown/ for inspection
from document_processing.extract import Document, write_markdown_doc

doc_meta = extract_pdf_meta(sample_pdf)

for name, text in {
    "basic": basic_text,
    "docling": docling_md,
    "pymupdf4llm": mu4llm_md,
}.items():
    doc = Document(
        source=Path(f"{sample_pdf.stem} [{name}]"),
        text=text,
        metadata=doc_meta,
    )
    out = write_markdown_doc(doc=doc, output_dir=MARKDOWN_DIR)
    print(f"{name:<12} -> {out.name}  ({out.stat().st_size} bytes)")

basic        -> Large Language Models the New Scrum Masters [basic].md  (23837 bytes)
docling      -> Large Language Models the New Scrum Masters [docling].md  (28367 bytes)
pymupdf4llm  -> Large Language Models the New Scrum Masters [pymupdf4llm].md  (24245 bytes)


### Metadata via `extract_pdf_meta`

The Markdown extractors return text only. `extract_pdf_meta` is the standalone, junk-filtered metadata reader — it drops filename / empty / `"Microsoft Word - "` titles and splits the author string into a list. Pair it with any extractor.

In [10]:
meta = extract_pdf_meta(sample_pdf)
print("metadata:", meta)

metadata: {'title': 'Large Language Models, the New Scrum Masters', 'authors': ['Juan Couder and Omar Ochoa']}


## Stage 2 — Section-aware chunking with LangChain

`01_demo` chunked by fixed character windows (`size=2000, overlap=200`) — that splits mid-sentence and ignores document structure. Here we split on Markdown **headings** with LangChain's `MarkdownHeaderTextSplitter`, so each chunk is a coherent section with its heading path attached as metadata.

We use the **docling** Markdown (richest heading tree); set `source_md = mu4llm_md` to compare.

In [11]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

source_md = docling_md  # try: source_md = mu4llm_md

headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]
splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
sections = splitter.split_text(source_md)

print(f"{len(sections)} sections\n")
for s in sections[:5]:
    crumb = " > ".join(s.metadata[h] for h in ("h1", "h2", "h3") if s.metadata.get(h))
    print(f"[{crumb or '(no heading)'}]  {len(s.page_content)} chars")
    print(f"  {s.page_content[:160].replace(chr(10), ' ')}...\n")

25 sections

[(no heading)]  69 chars
  <!-- image -->   [Papers](https://commons.erau.edu/red-papers)   2024...

[Large Language Models, the New Scrum Masters]  136 chars
  Juan Couder ortizcoj@my.erau.edu   Omar Ochoa ochoao@erau.edu   Follow this and additional works at: https://commons.erau.edu/red-papers...

[Scholarly Commons Citation]  618 chars
  Couder, J., &amp; Ochoa, O. (2024). Large Language Models, the New Scrum Masters. , (). Retrieved from https://commons.erau.edu/red-papers/2   This Paper is bro...

[[LARGE LANGUAGE MODELS, THE NEW SCRUM MASTERS](https://www.researchgate.net/publication/382382388_LARGE_LANGUAGE_MODELS_THE_NEW_SCRUM_MASTERS?enrichId=rgreq-61b2c96381af5db335670aed6100462f-XXX&enrichSource=Y292ZXJQYWdlOzM4MjM4MjM4ODtBUzoxMTQzMTI4MTI2MjU5MjQzOUAxNzIxNjQ2MjA5NTgz&el=1_x_3&_esc=publicationCoverPdf)]  14 chars
  <!-- image -->...

[J. Ortiz Couder, O. Ochoa]  52 chars
  Embry-Riddle Aeronautical University (UNITED STATES)...



In [15]:
sections[0]

Document(metadata={}, page_content='<!-- image -->  \n[Papers](https://commons.erau.edu/red-papers)  \n2024')

split each section

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
)

final_docs = char_splitter.split_documents(sections)
print(f"{len(final_docs)} final chunks\n")

62 final chunks



### Wrap sections as pipeline `Chunk`s

To stay compatible with the rest of the pipeline (embedder → retriever expect `Chunk`), map each LangChain section onto the project's `Chunk` dataclass. The section's heading path is preserved in `metadata["section"]`, alongside the doc's title/authors from `extract_pdf_meta`.

In [16]:
doc_meta = extract_pdf_meta(sample_pdf)

chunks = [
    Chunk(
        id=f"{sample_pdf.stem}#{i}",
        source=sample_pdf.name,
        index=i,
        text=s.page_content,
        metadata={
            **doc_meta,
            "section": " > ".join(
                s.metadata[h] for h in ("h1", "h2", "h3") if s.metadata.get(h)
            ),
        },
    )
    for i, s in enumerate(sections)
]

print(f"{len(chunks)} chunks\n")
first = chunks[0]
print("first chunk:")
print(f"  id:       {first.id}")
print(f"  source:   {first.source}")
print(f"  index:    {first.index}")
print(f"  metadata: {first.metadata}")
print(f"  text[:300]:\n{first.text[:300]}")

25 chunks

first chunk:
  id:       Large Language Models the New Scrum Masters#0
  source:   Large Language Models the New Scrum Masters.pdf
  index:    0
  metadata: {'title': 'Large Language Models, the New Scrum Masters', 'authors': ['Juan Couder and Omar Ochoa'], 'section': ''}
  text[:300]:
<!-- image -->  
[Papers](https://commons.erau.edu/red-papers)  
2024


In [17]:
chunks[0]

Chunk(id='Large Language Models the New Scrum Masters#0', source='Large Language Models the New Scrum Masters.pdf', index=0, text='<!-- image -->  \n[Papers](https://commons.erau.edu/red-papers)  \n2024', metadata={'title': 'Large Language Models, the New Scrum Masters', 'authors': ['Juan Couder and Omar Ochoa'], 'section': ''})

### Write chunks to `data/chunks/`

Same on-disk contract as `01_demo`'s `chunk_docs`: one `.jsonl` per source doc, one chunk per line. Downstream stages (embedder, retriever, answerer) consume this unchanged.

In [18]:
import json
from dataclasses import asdict

out = CHUNKS_DIR / f"{sample_pdf.stem}.jsonl"
with out.open("w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(asdict(c)) + "\n")

print(f"wrote {len(chunks)} chunks -> {out.name} ({out.stat().st_size} bytes)")

wrote 25 chunks -> Large Language Models the New Scrum Masters.jsonl (34701 bytes)


### Notes & next steps

- **Section chunks are uneven** — a long section can be huge, a heading-only section tiny. A common refinement is a second pass with `RecursiveCharacterTextSplitter` to cap section size while keeping the heading metadata. Left out here to keep "split by sections" literal.
- **Extractor choice drives chunk quality** — `extract_pdf_basic` has no headings, so it can't be section-split; docling usually yields the cleanest heading tree at the cost of speed.
- These `.jsonl` chunks are drop-in for the embedder → retriever → answerer stages shown in `01_demo`.